In [1]:
%gui qt
import numpy as np
from mayavi import mlab
import matplotlib.pyplot as plt

from kifmm_py import (
    KiFmm,
    # LaplaceKernel,
    HelmholtzKernel,
    SingleNodeTree,
    EvalType,
    BlasFieldTranslation,
    FftFieldTranslation,
lib
)

********************************************************************************
         to build the TVTK classes (9.2). This may cause problems.
         Please rebuild TVTK.
********************************************************************************



In [31]:
potentials = lib.leaf_potentials(fmm._fmm, leaf)

In [38]:
fmm.potential_dtype

numpy.complex64

In [35]:
potentials.data[0].len

21

In [37]:
fmm.coordinates_target_tree(leaf).shape

(63,)

In [28]:
np.random.seed(0)

dim = 3
dtype = np.float32
ctype = np.complex64

# Set FMM Parameters
expansion_order = np.array([10], np.uint64)  # Single expansion order as using n_crit
n_vec = 1
n_crit = 150
n_sources = 100000
n_targets = 100000
wavenumber = dtype(10)
prune_empty = True

# Setup source/target/charge data in Fortran order
sources = np.random.rand(n_sources * dim).astype(dtype)
targets = np.random.rand(n_targets * dim).astype(dtype)
charges = np.random.rand(n_sources * n_vec).astype(ctype)

eval_type = EvalType.Value

# EvalType computes either potentials (EvalType.Value) or potentials + derivatives (EvalType.ValueDeriv)
kernel = HelmholtzKernel(dtype, wavenumber, eval_type)

tree = SingleNodeTree(sources, targets, charges, n_crit=n_crit, prune_empty=prune_empty)

field_translation = FftFieldTranslation(kernel, block_size=32)

def error(direct, pot):
    return np.abs(direct) - np.abs(pot)

fmm = KiFmm(expansion_order, tree, field_translation, timed=True)

In [29]:
fmm.evaluate()

In [30]:
leaf = fmm.leaves_target_tree[0]

In [31]:
fmm.leaf_potentials(leaf)

array([[-201.52011-128.82053j  , -215.63171 -82.63208j  ,
        -213.37692 -88.78958j  , -237.3247  -41.706833j ,
        -247.10852 -60.455242j , -219.98375 -74.83475j  ,
        -252.12466 -44.85883j  , -244.41455 -58.75425j  ,
        -231.82332 -77.835j    , -213.35912 -93.915825j ,
        -262.2642   +7.6417007j, -246.19821 -35.79252j  ,
        -256.46088 -27.907936j , -256.81665  +3.6561165j,
        -208.05266 -96.83302j  , -240.75943 -40.829582j ,
        -217.821   -73.273506j , -227.76833 -90.05124j  ,
        -257.44888 -37.905224j , -221.89816 -95.55828j  ,
        -236.43959 -42.460114j ]], dtype=complex64)

In [32]:
leaf_coords = fmm.coordinates_target_tree(leaf)
nleaf_coords = len(leaf_coords) // 3

direct = np.zeros(nleaf_coords * eval_type.value).astype(ctype)
fmm.evaluate_kernel(EvalType.Value, sources, leaf_coords, charges, direct)

In [33]:
direct

array([-201.52225-128.82275j  , -215.63391 -82.63344j  ,
       -213.3785  -88.791725j , -237.32387 -41.70901j  ,
       -247.10812 -60.457745j , -219.98482 -74.83694j  ,
       -252.12169 -44.86092j  , -244.41522 -58.756702j ,
       -231.82428 -77.83724j  , -213.36067 -93.917946j ,
       -262.2629   +7.6400394j, -246.19884 -35.79368j  ,
       -256.4592  -27.911419j , -256.8161   +3.6553829j,
       -208.05396 -96.83558j  , -240.76024 -40.83112j  ,
       -217.8224  -73.27503j  , -227.77069 -90.05313j  ,
       -257.44565 -37.905937j , -221.89998 -95.560555j ,
       -236.4397  -42.46167j  ], dtype=complex64)

In [34]:
nleaf_coords

21

In [16]:
leaf_coords.shape

(63,)